# SESSION 2.1: TENSORS & OPERATIONS — THEORY

---

## **WHAT IS A TENSOR?**

### **Simple Definition**
A tensor is a **multi-dimensional array** of numbers. Think of it as a generalization:

- **0D Tensor (Scalar):** Just a single number → `5`
- **1D Tensor (Vector):** A list of numbers → `[1, 2, 3, 4]`
- **2D Tensor (Matrix):** A table of numbers → `[[1, 2], [3, 4]]`
- **3D Tensor:** A cube of numbers → Think of a video frame (height × width × color channels)
- **4D Tensor:** A batch of video frames → (batch × height × width × channels)

### **Visual Example**
```
Scalar (0D):
    5

Vector (1D):
    [1, 2, 3, 4, 5]

Matrix (2D):
    [[1, 2, 3],
     [4, 5, 6]]

3D Tensor:
    [[[1, 2],    First "slice"
      [3, 4]],
     
     [[5, 6],    Second "slice"
      [7, 8]]]
```

---

## **TENSORS IN AUDIO PROCESSING**

### **Your Audio Data as Tensors**

When you load audio with librosa, you get a **1D numpy array**:
```python
audio = [0.01, 0.02, -0.03, 0.04, ...]  # 16,000 samples = 1 second
# Shape: (16000,)
```

But neural networks need **batched, multi-channel data**:
```python
# Single audio sample for training:
Shape: (1, 1, 16000)
       │  │    └── samples (time dimension)
       │  └── channels (mono = 1, stereo = 2)
       └── batch size (1 audio clip)

# Batch of 32 audio samples:
Shape: (32, 1, 16000)
       └── 32 different audio clips processed together
```

### **Why Batching?**
- **Efficiency:** Process multiple samples simultaneously using GPU parallelism
- **Stable gradients:** Average gradients across batch reduces noise
- **Faster training:** 32 samples at once is much faster than 32 × 1

---

## **TENSOR vs NUMPY ARRAY**

### **Similarities**
- Both are multi-dimensional arrays
- Both support indexing, slicing, reshaping
- Both support mathematical operations

### **Key Differences**

| Feature | NumPy Array | PyTorch Tensor |
|---------|-------------|----------------|
| **Device** | CPU only | CPU or GPU |
| **Gradients** | No automatic differentiation | Built-in autograd |
| **Deep Learning** | Not designed for it | Optimized for neural networks |
| **Speed** | Fast on CPU | Fast on both CPU/GPU |

### **Why PyTorch for Deep Learning?**

**1. Automatic Differentiation (Autograd)**
```python
# PyTorch tracks operations for automatic gradient computation
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2  # y = 4
y.backward()  # Compute dy/dx automatically
print(x.grad)  # Output: tensor([4.0])  because dy/dx = 2x = 2(2) = 4
```

NumPy can't do this! You'd have to manually compute gradients.

**2. GPU Acceleration**
```python
# Move tensor to GPU (if available)
tensor_gpu = tensor.to('cuda')
# Now all operations are 10-100x faster!
```

---

## **TENSOR SHAPES & DIMENSIONS**

### **Understanding Shape**

```python
tensor = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])

print(tensor.shape)  # torch.Size([2, 3])
                     #              │  └── 3 columns
                     #              └── 2 rows

print(tensor.dim())  # 2 (it's a 2D tensor)
print(tensor.size(0))  # 2 (size of dimension 0)
print(tensor.size(1))  # 3 (size of dimension 1)
```

### **Audio Example**
```python
# Single mono audio clip (1 second at 16kHz)
audio = torch.randn(16000)
print(audio.shape)  # torch.Size([16000])
print(audio.dim())  # 1 (1D tensor)

# Reshape for neural network input
audio_batched = audio.view(1, 1, 16000)
print(audio_batched.shape)  # torch.Size([1, 1, 16000])
print(audio_batched.dim())  # 3 (3D tensor)
```

---

## **COMMON TENSOR OPERATIONS**

### **1. Reshaping**

Changing shape without changing data:

```python
x = torch.tensor([1, 2, 3, 4, 5, 6])  # Shape: (6,)

# Reshape to 2x3 matrix
x_2d = x.view(2, 3)
# [[1, 2, 3],
#  [4, 5, 6]]

# Reshape to 3x2 matrix
x_3x2 = x.view(3, 2)
# [[1, 2],
#  [3, 4],
#  [5, 6]]

# Use -1 to auto-calculate one dimension
x_auto = x.view(2, -1)  # Same as view(2, 3)
```

**Important:** `view()` requires contiguous memory. Use `reshape()` if not sure.

### **2. Adding Dimensions (Unsqueeze)**

Adding a dimension of size 1:

```python
x = torch.tensor([1, 2, 3])  # Shape: (3,)

x_unsqueeze_0 = x.unsqueeze(0)  # Shape: (1, 3)
# [[1, 2, 3]]  ← Now it's a 2D tensor

x_unsqueeze_1 = x.unsqueeze(1)  # Shape: (3, 1)
# [[1],
#  [2],
#  [3]]  ← Also 2D, but different shape
```

**Audio Example:**
```python
audio = torch.randn(16000)  # Shape: (16000,)

# Add batch dimension
audio = audio.unsqueeze(0)  # Shape: (1, 16000)

# Add channel dimension
audio = audio.unsqueeze(0)  # Shape: (1, 1, 16000)
# Now ready for Conv1d!
```

### **3. Removing Dimensions (Squeeze)**

Removing dimensions of size 1:

```python
x = torch.tensor([[[1, 2, 3]]])  # Shape: (1, 1, 3)

x_squeezed = x.squeeze()  # Shape: (3,)
# [1, 2, 3]  ← All size-1 dimensions removed
```

### **4. Concatenation**

Joining tensors along a dimension:

```python
audio1 = torch.randn(1, 1, 8000)  # 0.5 seconds
audio2 = torch.randn(1, 1, 8000)  # 0.5 seconds

# Concatenate along time dimension (dim=2)
combined = torch.cat([audio1, audio2], dim=2)
print(combined.shape)  # torch.Size([1, 1, 16000])
# Now you have 1 second of audio!
```

---

## **GRADIENTS & AUTOGRAD**

This is what makes PyTorch powerful for deep learning!

### **The Concept**

In machine learning, we need to compute **how loss changes with respect to each parameter**:

```
If Loss = f(weight), then we need: ∂Loss/∂weight
```

This tells us which direction to adjust weights to reduce loss.

### **Manual vs Automatic**

**Manual (painful):**
```python
# You have to derive and code the gradient formula
def forward(x, w):
    return x * w

def backward(x, w):
    # Manually computed: d(x*w)/dw = x
    return x  

# For complex networks, this is VERY hard!
```

**Automatic (PyTorch):**
```python
x = torch.tensor([2.0])
w = torch.tensor([3.0], requires_grad=True)  # Track gradients

y = x * w  # Forward pass: y = 6.0
y.backward()  # Compute gradients automatically

print(w.grad)  # tensor([2.0])  ← PyTorch computed this!
```

### **How Autograd Works (Simplified)**

PyTorch builds a **computational graph** tracking all operations:

```
        x (2.0)    w (3.0, requires_grad=True)
           │         │
           └────*────┘  ← Multiplication operation
                │
              y (6.0)
```

When you call `y.backward()`:
1. Start at y
2. Walk backward through the graph
3. Apply chain rule at each operation
4. Accumulate gradients in `.grad` attributes

### **Important Gradient Concepts**

**1. requires_grad=True**
```python
# Only tensors with requires_grad=True track gradients
x = torch.tensor([1.0])  # requires_grad=False (default)
w = torch.tensor([2.0], requires_grad=True)  # Will track gradients

y = x * w
# y inherits requires_grad=True because w has it
```

**2. Gradient Accumulation**
```python
w = torch.tensor([1.0], requires_grad=True)

y1 = w * 2
y1.backward()
print(w.grad)  # tensor([2.0])

y2 = w * 3
y2.backward()
print(w.grad)  # tensor([5.0])  ← Accumulated! (2 + 3)
```

**3. Zero Gradients**
```python
# Clear gradients before next backward pass
w.grad.zero_()

y3 = w * 4
y3.backward()
print(w.grad)  # tensor([4.0])  ← Fresh gradient
```

This is crucial in training loops!

---

## **TENSOR DATA TYPES**

### **Common Types**

```python
# Integer types
torch.int32      # 32-bit integer
torch.int64      # 64-bit integer (torch.long)

# Float types
torch.float32    # 32-bit float (torch.float) - MOST COMMON
torch.float64    # 64-bit float (torch.double)

# Boolean
torch.bool       # True/False
```

### **Why float32?**
- ✅ Sufficient precision for neural networks
- ✅ Half the memory of float64
- ✅ Faster computation
- ✅ Standard in deep learning

### **Type Conversion**
```python
x = torch.tensor([1, 2, 3])  # Default: int64
x_float = x.float()          # Convert to float32
x_double = x.double()        # Convert to float64
```

---

## **CPU vs GPU (CUDA)**

### **Device Management**

```python
# Check if GPU is available
print(torch.cuda.is_available())  # True if you have NVIDIA GPU

# Create tensor on CPU (default)
x_cpu = torch.tensor([1, 2, 3])

# Move to GPU
if torch.cuda.is_available():
    x_gpu = x_cpu.to('cuda')
    # or
    x_gpu = x_cpu.cuda()
    
# Move back to CPU
x_back = x_gpu.to('cpu')
# or
x_back = x_gpu.cpu()
```

### **Important Rules**
- ⚠️ **All tensors in an operation must be on the same device**
- ⚠️ **NumPy only works with CPU tensors**

```python
# This will error:
x_cpu = torch.tensor([1, 2])
x_gpu = torch.tensor([3, 4]).cuda()
result = x_cpu + x_gpu  # ❌ Error! Different devices

# Fix: Move both to same device
result = x_cpu.cuda() + x_gpu  # ✅ Works
```

---

## **KEY TAKEAWAYS**

1. **Tensors are multi-dimensional arrays** optimized for deep learning
2. **Shape matters:** Neural networks expect specific tensor shapes
3. **Batching:** Process multiple samples together for efficiency
4. **Autograd:** PyTorch automatically computes gradients for backpropagation
5. **float32:** Standard data type for neural networks
6. **Device consistency:** Keep tensors on same device (CPU or GPU)

---

## **AUDIO-SPECIFIC SUMMARY**

For your voice isolation project:

```python
# Input: NumPy array from librosa
audio_numpy = librosa.load(...)  # Shape: (16000,)

# Convert to tensor
audio_tensor = torch.from_numpy(audio_numpy)  # Shape: (16000,)

# Reshape for Conv1d input: (batch, channels, samples)
audio_tensor = audio_tensor.unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, 16000)

# Now ready for your model!
```

**EXCELLENT QUESTIONS!** 🎯

Let me answer both before giving you the tasks.

---

## **QUESTION 1: RESHAPING RULE**

### **YES! You're exactly right!**

**The Rule:**
```
Total number of elements must remain the same
```

**Math:**
```python
original: (6,)     → 6 elements total
reshape:  (2, 3)   → 2 × 3 = 6 elements ✅
reshape:  (3, 2)   → 3 × 2 = 6 elements ✅
reshape:  (1, 6)   → 1 × 6 = 6 elements ✅
reshape:  (6, 1)   → 6 × 1 = 6 elements ✅

# This would ERROR:
reshape:  (2, 2)   → 2 × 2 = 4 elements ❌ (6 ≠ 4)
```

**Audio Example:**
```python
audio = torch.randn(16000)  # 16,000 elements

# Valid reshapes:
audio.view(1, 16000)        # 1 × 16,000 = 16,000 ✅
audio.view(1, 1, 16000)     # 1 × 1 × 16,000 = 16,000 ✅
audio.view(2, 8000)         # 2 × 8,000 = 16,000 ✅
audio.view(100, 160)        # 100 × 160 = 16,000 ✅

# Invalid:
audio.view(1, 10000)        # 1 × 10,000 = 10,000 ❌ ERROR!
```

**The -1 Trick:**
```python
# Use -1 to auto-calculate one dimension
audio.view(1, 1, -1)   # PyTorch calculates: -1 = 16000/1/1 = 16000
audio.view(2, -1)      # PyTorch calculates: -1 = 16000/2 = 8000
audio.view(-1, 160)    # PyTorch calculates: -1 = 16000/160 = 100
```

---

## **QUESTION 2: GPU HANDLING**

### **A. Different GPU Types**

**NVIDIA GPUs (CUDA):**
```python
device = torch.device('cuda')  # NVIDIA GPU
```

**Apple Silicon (M1/M2/M3) - MPS:**
```python
device = torch.device('mps')   # Apple Metal Performance Shaders
```

**AMD GPUs (ROCm):**
```python
device = torch.device('cuda')  # AMD with ROCm (same API as NVIDIA)
```

**CPU (Fallback):**
```python
device = torch.device('cpu')
```

---

### **B. Automatic Device Selection**

**Smart Device Selection:**
```python
# Automatically choose best available device
if torch.cuda.is_available():
    device = torch.device('cuda')      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device('mps')       # Apple Silicon
else:
    device = torch.device('cpu')       # Fallback

print(f"Using device: {device}")
```

---

### **C. Default Tensor Creation on GPU**

**Method 1: Set Default Device (Explicit)**
```python
# NOT a built-in PyTorch feature, but you can create a helper:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def tensor(*args, **kwargs):
    """Create tensor on default device"""
    return torch.tensor(*args, **kwargs).to(device)

# Now use your custom function:
x = tensor([1, 2, 3])  # Created on GPU automatically!
print(x.device)  # cuda:0
```

**Method 2: Wrapper Class (Cleaner)**
```python
class TensorFactory:
    def __init__(self):
        if torch.cuda.is_available():
            self.device = torch.device('cuda')
        elif torch.backends.mps.is_available():
            self.device = torch.device('mps')
        else:
            self.device = torch.device('cpu')
    
    def tensor(self, *args, **kwargs):
        return torch.tensor(*args, **kwargs, device=self.device)
    
    def randn(self, *args, **kwargs):
        return torch.randn(*args, **kwargs, device=self.device)
    
    def zeros(self, *args, **kwargs):
        return torch.zeros(*args, **kwargs, device=self.device)

# Usage:
tf = TensorFactory()
x = tf.tensor([1, 2, 3])      # On GPU automatically
y = tf.randn(100, 100)        # On GPU automatically
z = tf.zeros(10, 10)          # On GPU automatically
```

**Method 3: PyTorch's Built-in `set_default_tensor_type()` (Not Recommended)**
```python
# This changes the DEFAULT type globally
torch.set_default_tensor_type('torch.cuda.FloatTensor')

# Now tensors are created on GPU by default
x = torch.tensor([1, 2, 3])  # On GPU!

# ⚠️ Problem: This can cause unexpected behavior in libraries
# ⚠️ Not recommended for production code
```

---

### **D. Best Practice (What I Recommend)**

**Create a config at the top of your notebook/script:**

```python
# At the start of your notebook
import torch

# Device configuration
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ Using NVIDIA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device('cpu')
    print("⚠️  Using CPU (no GPU available)")

# Helper function for easy tensor creation
def to_device(tensor):
    return tensor.to(device)

# Usage throughout your code:
x = torch.tensor([1, 2, 3]).to(device)
# or
x = to_device(torch.tensor([1, 2, 3]))

# For model:
model = MyModel().to(device)
```

---

### **E. For Your MacBook Specifically**

If you have Apple Silicon (M1/M2/M3):

```python
import torch

# Check MPS availability
if torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("❌ MPS not available because PyTorch not built with MPS")
    else:
        device = torch.device("mps")
        print("✅ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠️  MPS not available, using CPU")

# Test it
x = torch.randn(5, 5).to(device)
print(x.device)  # mps:0 or cpu
```

**Note for Apple Silicon:**
- MPS support was added in PyTorch 1.12+
- Not all operations are supported on MPS yet
- Some operations automatically fall back to CPU

---

### **F. Summary: Device Management Strategy**

```python
# ✅ RECOMMENDED PATTERN:

# 1. Set device once at the top
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Move data to device when loading
for batch in dataloader:
    audio, labels = batch
    audio = audio.to(device)
    labels = labels.to(device)
    
    # Now everything is on the same device

# 3. Move model to device
model = MyModel().to(device)

# 4. All operations happen on the same device automatically
output = model(audio)  # Both model and audio on GPU → fast!
```

In [3]:
# ============================================
# SESSION 2.1: TENSORS & OPERATIONS
# Part A: Tensor Basics
# ============================================

import torch
import numpy as np
from pathlib import Path

# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


## TODO 1: Create Your First Tensor

In [4]:
# TODO 1.1: Create a 1D tensor with values [1, 2, 3, 4, 5]
# Hint: Use torch.tensor()

tensor_1d = torch.tensor([1, 2, 3, 4, 5])

# Test your answer
print("TODO 1.1:")
print(f"Tensor: {tensor_1d}")
print(f"Shape: {tensor_1d.shape}")
print(f"Dimensions: {tensor_1d.dim()}")
print(f"Data type: {tensor_1d.dtype}")

# Expected output:
# Tensor: tensor([1, 2, 3, 4, 5])
# Shape: torch.Size([5])
# Dimensions: 1
# Data type: torch.int64

TODO 1.1:
Tensor: tensor([1, 2, 3, 4, 5])
Shape: torch.Size([5])
Dimensions: 1
Data type: torch.int64


In [5]:
# TODO 1.2: Create a 2D tensor (matrix) with shape (3, 4) containing random values
# Hint: Use torch.randn() which creates random values from normal distribution

tensor_2d = torch.randn([3,4])

# Test your answer
print("\nTODO 1.2:")
print(f"Tensor:\n{tensor_2d}")
print(f"Shape: {tensor_2d.shape}")
print(f"Dimensions: {tensor_2d.dim()}")

# Expected output:
# Shape: torch.Size([3, 4])
# Dimensions: 2


TODO 1.2:
Tensor:
tensor([[-0.8709,  0.9305, -0.2655, -2.0716],
        [ 1.1652, -1.5406,  1.4173,  0.4881],
        [ 0.6678,  1.6502, -1.0658, -0.6943]])
Shape: torch.Size([3, 4])
Dimensions: 2


In [6]:
# TODO 1.3: Create a tensor of zeros with shape (2, 3, 4)
# Hint: Use torch.zeros()

tensor_zeros = torch.zeros([2,3,4])

# Test your answer
print("\nTODO 1.3:")
print(f"Shape: {tensor_zeros.shape}")
print(f"All zeros? {torch.all(tensor_zeros == 0)}")

# Expected: Shape: torch.Size([2, 3, 4])
# Expected: All zeros? True


TODO 1.3:
Shape: torch.Size([2, 3, 4])
All zeros? True


## TODO 2: NumPy ↔ PyTorch Conversion

In [11]:
# TODO 2.1: Convert a NumPy array to PyTorch tensor
numpy_array = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

# TODO: Convert to tensor
# Hint: Use torch.from_numpy()
tensor_from_numpy = torch.from_numpy(numpy_array)

# Test your answer
print("\nTODO 2.1:")
# if numpy is not converted to tensor then rase a error
assert isinstance(tensor_from_numpy, torch.Tensor), "Should be a PyTorch tensor!"
print(f"Original NumPy type: {type(numpy_array)}")
print(f"Converted type: {type(tensor_from_numpy)}")
print(f"Values match: {np.array_equal(numpy_array, tensor_from_numpy.numpy())}")




TODO 2.1:
Original NumPy type: <class 'numpy.ndarray'>
Converted type: <class 'torch.Tensor'>
Values match: True


In [14]:
# TODO 2.2: Convert a PyTorch tensor back to NumPy
torch_tensor = torch.tensor([10, 20, 30, 40, 50])

# TODO: Convert to numpy
# Hint: Use .numpy() method
numpy_from_torch = torch_tensor.numpy()

# Test your answer
print("\nTODO 2.2:")
assert isinstance(numpy_from_torch, np.ndarray), "Should be a NumPy array!"
print(f"Tensor type: {type(torch_tensor)}")
print(f"NumPy type: {type(numpy_from_torch)}")
print(f"Values: {numpy_from_torch}")


TODO 2.2:
Tensor type: <class 'torch.Tensor'>
NumPy type: <class 'numpy.ndarray'>
Values: [10 20 30 40 50]
